In [0]:
df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("abfss://companydocuments@stcorestoragepy2026.dfs.core.windows.net/orders.csv")
)

display(df)


order_id,customer_id,order_date,product,quantity,unit_price_usd
1001,C001,2026-01-10,Laptop,1,1200
1002,C002,2026-01-11,Monitor,2,300
1003,C003,2026-01-12,Keyboard,1,80
1004,C001,2026-01-13,Mouse,3,25
1005,C004,2026-01-14,Docking Station,1,180


In [0]:
df.withColumn("total_amount", df.quantity * df.unit_price_usd).groupBy("customer_id").sum("total_amount").display()

customer_id,sum(total_amount)
C003,80
C004,180
C001,1275
C002,600


In [0]:
# 2) Simple transformation (example: add total_amount column)
from pyspark.sql.functions import col

df_transformed = df.withColumn(
    "total_amount_usd",
    col("quantity") * col("unit_price_usd")
)

# 3) Write transformed data to a new folder in Blob Storage
df_transformed.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("abfss://companydocuments@stcorestoragepy2026.dfs.core.windows.net/processed/orders_enriched")

# 4) (Optional) Read it back to verify
df_check = spark.read.option("header", "true").csv(
    "abfss://companydocuments@stcorestoragepy2026.dfs.core.windows.net/processed/orders_enriched"
)

display(df_check)

order_id,customer_id,order_date,product,quantity,unit_price_usd,total_amount_usd
1001,C001,2026-01-10,Laptop,1,1200,1200
1002,C002,2026-01-11,Monitor,2,300,600
1003,C003,2026-01-12,Keyboard,1,80,80
1004,C001,2026-01-13,Mouse,3,25,75
1005,C004,2026-01-14,Docking Station,1,180,180


In [0]:
# Read previously written data from Azure Blob Storage
df_read = spark.read.option("header", "true").csv(
    "abfss://companydocuments@stcorestoragepy2026.dfs.core.windows.net/processed/orders_enriched"
)

# Register DataFrame as a temporary SQL view
df_read.createOrReplaceTempView("orders_enriched")

# Query data using Spark SQL to aggregate total spend per customer
spark.sql("""
    SELECT customer_id, SUM(total_amount_usd) AS total_spend
    FROM orders_enriched
    GROUP BY customer_id
""").display()


customer_id,total_spend
C003,80.0
C004,180.0
C001,1275.0
C002,600.0


In [0]:
from pyspark.sql.functions import col

# Read previously written enriched orders
df_read = spark.read.option("header", "true").csv(
    "abfss://companydocuments@stcorestoragepy2026.dfs.core.windows.net/processed/orders_enriched"
)

# Cast column to numeric BEFORE aggregation
df_read = df_read.withColumn(
    "total_amount_usd",
    col("total_amount_usd").cast("float")
)

# Aggregate total spend per customer
df_result = (
    df_read
    .groupBy("customer_id")
    .sum("total_amount_usd")
    .withColumnRenamed("sum(total_amount_usd)", "total_spend_usd")
)

# Write result as Delta table
df_result.write \
    .format("delta") \
    .mode("overwrite") \
    .save("abfss://companydocuments@stcorestoragepy2026.dfs.core.windows.net/delta/customer_totals")

# Read Delta table back to verify
df_delta = spark.read.format("delta").load(
    "abfss://companydocuments@stcorestoragepy2026.dfs.core.windows.net/delta/customer_totals"
)

display(df_delta)


customer_id,total_spend_usd
C003,80.0
C004,180.0
C001,1275.0
C002,600.0


In [0]:
# Read Delta table with customer totals
df_customers = spark.read.format("delta").load(
    "abfss://companydocuments@stcorestoragepy2026.dfs.core.windows.net/delta/customer_totals"
)

# Show all customers ordered by total spend
df_customers_ordered = df_customers.orderBy("total_spend_usd", ascending=False)

# Display result (analytics-ready)
display(df_customers_ordered)


customer_id,total_spend_usd
C001,1275.0
C002,600.0
C004,180.0
C003,80.0


Databricks visualization. Run in Databricks to view.

In [0]:
%%sql
CREATE DATABASE IF NOT EXISTS demo
LOCATION 'abfss://companydocuments@stcorestoragepy2026.dfs.core.windows.net/metastore';

CREATE TABLE IF NOT EXISTS demo.customer_totals
USING DELTA
LOCATION 'abfss://companydocuments@stcorestoragepy2026.dfs.core.windows.net/delta/customer_totals';



In [0]:
%%sql
SELECT * FROM demo.customer_totals;


customer_id,total_spend_usd
C003,80.0
C004,180.0
C001,1275.0
C002,600.0


DataFrame[customer_id: string, total_spend_usd: double]